# Setup and orientation

Assistant version 0: Make a first call and inspect an optional trace.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --group authoring`, then `uv run jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## LLMs, tokens, context windows, temperature, hallucinations

Predict the output before running. The function definitions below match `examples/01_llm_basics.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare supplied evidence with an unknown fact; tokens are pieces of text.

Run: uv run python modules/00_setup/examples/01_llm_basics.py --offline
Expected: A grounded return-window reply; unknown private facts require evidence.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from langchain_core.messages import HumanMessage, SystemMessage

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message, print_text


def context_fits(input_tokens: int, output_reserve: int, context_window: int) -> bool:
    if min(input_tokens, output_reserve, context_window) < 0:
        raise ValueError("Token counts cannot be negative.")
    return input_tokens + output_reserve <= context_window


def main() -> None:
    settings = lesson_settings()
    # KEY IDEA: these are supplied counts, not an inaccurate character-based tokenizer.
    print_text(f"120 input + 40 reserved fit a 256-token window: {context_fits(120, 40, 256)}")
    model = get_chat_model(settings=settings, temperature=0)
    print_message(
        model.invoke(
            [
                SystemMessage(
                    "Answer from this evidence only: unused gear can be returned within "
                    "30 days of delivery with proof of purchase. Say unknown if absent."
                ),
                HumanMessage(
                    "What is the return window? What is my private order's delivery date?"
                ),
            ]
        )
    )
    print_text("Temperature changes sampling, not truth. A fluent answer still needs checking.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## LangChain, core, provider packages, LangGraph, LangSmith, Deep Agents; choose a building block

Predict the output before running. The function definitions below match `examples/02_ecosystem.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Choose the smallest building block for a Brightwave task.

Run: uv run python modules/00_setup/examples/02_ecosystem.py --offline
Expected: Draft -> model; fixed pipeline -> LCEL; durable approval -> LangGraph.
Cost: No model calls.
"""

from importlib.metadata import version

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def choose_block(
    *, fixed_steps: bool = False, durable: bool = False, tool_decisions: bool = False
) -> str:
    if durable:
        return "LangGraph"
    if tool_decisions:
        return "LangChain agent"
    return "LCEL" if fixed_steps else "model"


def main() -> None:
    lesson_settings()
    for package in ("langchain", "langchain-core", "langchain-openai", "langgraph", "langsmith"):
        print_text(f"{package}: {version(package)}")
    # KEY IDEA: a bigger framework is useful only when the task needs its behavior.
    print_text(
        f"Draft: {choose_block()}; fixed pipeline: {choose_block(fixed_steps=True)}; "
        f"durable approval: {choose_block(durable=True)}"
    )
    print_text(
        "core defines interfaces; providers translate requests; LangSmith observes; "
        "Deep Agents adds planning/filesystem patterns in an optional later tier."
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## uv, pip fallback, virtual environments, .env, package and key checks, Ollama pull, optional tracing

Predict the output before running. The function definitions below match `examples/03_verify_environment.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Check the environment without a billable model request.

Run: uv run python modules/00_setup/examples/03_verify_environment.py --offline
Expected: Package versions and OFFLINE CHECK PASSED.
Cost: No generation; live Ollama checks only its installed-model list.
"""

import runpy
import sys

from common.config import ROOT
from common.errors import run_cli
from common.lesson import lesson_settings


def main() -> None:
    settings = lesson_settings()
    old = sys.argv
    try:
        sys.argv = ["check_env.py", *(["--offline"] if settings.course_offline else [])]
        # KEY IDEA: checking credentials locally does not validate a hosted API key.
        runpy.run_path(str(ROOT / "scripts/check_env.py"), run_name="__main__")
    finally:
        sys.argv = old


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Make a first call and inspect an optional trace

Predict the output before running. The function definitions below match `use_case/first_call.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Make the first evidence-grounded Brightwave call with optional tracing.

Run: uv run python modules/00_setup/use_case/first_call.py --offline
Expected: A reply grounded in the return policy; tracing remains optional.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from langchain_core.messages import HumanMessage, SystemMessage

from common.data import data_path
from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message, print_usage
from common.tracing import tracing_scope


def main() -> None:
    settings = lesson_settings()
    policy = data_path("policies/returns.md").read_text(encoding="utf-8")
    with tracing_scope(settings):
        reply = get_chat_model(settings=settings).invoke(
            [
                SystemMessage("Use only this Brightwave policy as evidence:\n" + policy),
                HumanMessage("Can I return unused gear delivered 12 days ago with a receipt?"),
            ]
        )
    print_message(reply)
    print_usage(reply, settings=settings)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run pytest modules/00_setup/tests` for the supplied solution checks.